# 05 - Modelo Final e Predição em App

**Objetivo:** Treinar o modelo final na base completa e gerar as predições no `abalone_app.csv`.

**Modelo final escolhido:** SVM (RBF)


In [2]:
import os
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, f1_score
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

DATA_DIR = '../'
OUTPUT_DIR = './pipeline_output/'

df_full = pd.read_csv('/home/antonio/Documentos/Machine Learning/mlclass/03_Validation/abalone_dataset.csv')
df_app = pd.read_csv('/home/antonio/Documentos/Machine Learning/mlclass/03_Validation/abalone_app.csv')


best_name_file = OUTPUT_DIR + 'best_model_name.txt'
if os.path.exists(best_name_file):
    with open(best_name_file, 'r', encoding='utf-8') as f:
        MODEL_NAME = f.read().strip()
else:
    MODEL_NAME = 'SVM (RBF)'

print('Modelo escolhido:', MODEL_NAME)
print('Treino completo:', df_full.shape)
print('Aplicação:', df_app.shape)


Modelo escolhido: SVM (RBF)
Treino completo: (3132, 9)
Aplicação: (1045, 8)


## 1. Função para Criar o Modelo


In [3]:
feature_cols = ['length', 'diameter', 'height', 'whole_weight',
                'shucked_weight', 'viscera_weight', 'shell_weight']
cat_cols = ['sex']
num_cols = feature_cols

def make_one_hot():
    try:
        return OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)
    except TypeError:
        return OneHotEncoder(drop='first', handle_unknown='ignore', sparse=False)

def build_model(name):
    if name == 'Dummy (most frequent)':
        return DummyClassifier(strategy='most_frequent')
    if name == 'Logistic Regression':
        return LogisticRegression(max_iter=2000, random_state=42)
    if name == 'LDA':
        return LinearDiscriminantAnalysis()
    if name == 'KNN (k=5)':
        return KNeighborsClassifier(n_neighbors=5)
    if name == 'SVM (Linear)':
        return SVC(kernel='linear', random_state=42)
    if name == 'SVM (RBF)':
        return SVC(kernel='rbf', random_state=42)
    if name == 'Decision Tree':
        return DecisionTreeClassifier(random_state=42)
    if name == 'Random Forest (100)':
        return RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    return SVC(kernel='rbf', random_state=42)

model = build_model(MODEL_NAME)
preprocessor = ColumnTransformer([
    ('cat', make_one_hot(), cat_cols),
    ('num', StandardScaler(), num_cols),
])
final_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', model),
])

print(final_pipeline)


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['sex']),
                                                 ('num', StandardScaler(),
                                                  ['length', 'diameter',
                                                   'height', 'whole_weight',
                                                   'shucked_weight',
                                                   'viscera_weight',
                                                   'shell_weight'])])),
                ('classifier', SVC(random_state=42))])


## 2. Validação Cruzada Final


In [4]:
X_full = df_full.drop('type', axis=1)
y_full = df_full['type']
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

acc_scores = cross_val_score(final_pipeline, X_full, y_full, cv=cv, scoring='accuracy', n_jobs=-1)
f1_scores = cross_val_score(final_pipeline, X_full, y_full, cv=cv, scoring='f1_macro', n_jobs=-1)

print(f'Accuracy CV: {acc_scores.mean():.4f} +/- {acc_scores.std():.4f}')
print(f'Macro F1 CV: {f1_scores.mean():.4f} +/- {f1_scores.std():.4f}')


Accuracy CV: 0.6660 +/- 0.0164
Macro F1 CV: 0.6624 +/- 0.0166


## 3. Treino Final com Toda a Base


In [5]:
final_pipeline.fit(X_full, y_full)
print('Modelo final treinado com a base completa.')


Modelo final treinado com a base completa.


## 4. Predição no Dataset de Aplicação


In [6]:
predictions = final_pipeline.predict(df_app)
print('Total de predições:', len(predictions))
print(pd.Series(predictions).value_counts().sort_index())


Total de predições: 1045
1    325
2    317
3    403
Name: count, dtype: int64


## 5. Geração do Arquivo de Submissão


In [7]:
submission = pd.DataFrame({'type': predictions})
submission_path = OUTPUT_DIR + 'abalone_submission.csv'
submission.to_csv(submission_path, index=False)
print('Arquivo salvo em:', submission_path)
print(submission.head())


Arquivo salvo em: ./pipeline_output/abalone_submission.csv
   type
0     3
1     2
2     3
3     3
4     3


## 6. Salvamento do Pipeline Final


In [8]:
joblib.dump(final_pipeline, OUTPUT_DIR + 'final_pipeline.pkl')
print('Pipeline final salvo em:', OUTPUT_DIR + 'final_pipeline.pkl')


Pipeline final salvo em: ./pipeline_output/final_pipeline.pkl


## 7. Checklist Final

- [ ] Melhor modelo definido no notebook 04
- [ ] Validação local executada
- [ ] Arquivo de submissão gerado
- [ ] Quantidade de linhas conferida
- [ ] Pipeline final salvo


In [9]:
print('=' * 60)
print('RESUMO FINAL')
print('=' * 60)
print('Modelo:', MODEL_NAME)
print(f'Accuracy CV: {acc_scores.mean():.4f} +/- {acc_scores.std():.4f}')
print(f'Macro F1 CV: {f1_scores.mean():.4f} +/- {f1_scores.std():.4f}')
print(f'Predições geradas: {len(predictions)}')
print('Submissão pronta em:', submission_path)
print('=' * 60)


RESUMO FINAL
Modelo: SVM (RBF)
Accuracy CV: 0.6660 +/- 0.0164
Macro F1 CV: 0.6624 +/- 0.0166
Predições geradas: 1045
Submissão pronta em: ./pipeline_output/abalone_submission.csv
